# English → Persian Religious Text Translation

Fine-tuning `facebook/mbart-large-50-many-to-many-mmt` for English-to-Persian
translation of religious/doctrinal text.

**Pipeline:** scrape paired EN/FA pages → strictly align sentence pairs →
fine-tune mBART-50 → evaluate with BLEU.


## 1. Setup

Install dependencies.

In [1]:
!pip install -q requests beautifulsoup4 huggingface_hub evaluate
!pip install -q --upgrade transformers datasets


In [2]:
import transformers
print("transformers version:", transformers.__version__)


transformers version: 5.14.1


## 2. Data Collection

Scrape paired English/Persian pages and produce a strictly-aligned parallel
corpus of sentence pairs.

**Alignment strategy:**
- An entire document is discarded if its English and Persian paragraph
  counts don't match.
- Within an accepted document, each paragraph is discarded unless its
  English and Persian sentence counts match exactly.

This trades corpus size for alignment quality — mismatches are dropped
rather than approximated.


In [3]:
import requests
from bs4 import BeautifulSoup
import json
import re

# ======================================================
# *** CRITICAL: Update your variables here ***
CONTENT_SELECTOR = ".body-block"
OUTPUT_FILE = 'DATASET.jsonl'
PARAGRAPH_TAG = 'p' # Confirmed tag for body text
# ======================================================

# --- HELPER FUNCTIONS ---

def clean_text(text):
    """Normalize text and remove excessive whitespace/newlines."""
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def split_text_into_sentences(text, lang='en'):
    """Segments text into sentences based on punctuation."""
    if not text: return []
    if lang == 'fa':
        # Split on Persian/Arabic end-of-sentence punctuation (. ! ? ؛ ؟)
        sentences = re.split(r'(?<=[.?!؛؟])\s+', text)
    else:
        # Split on standard punctuation (. ! ?)
        sentences = re.split(r'(?<=[.?!])\s+', text)

    return [s.strip() for s in sentences if s.strip()]

def scrape_url_by_elements(url, selector, paragraph_tag='p'):
    """
    Fetches text from a single URL and extracts the content as a list of paragraphs
    by finding all elements matching the paragraph_tag within the main selector.
    """
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        # 1. Find the main container element
        main_element = soup.select_one(selector)

        if main_element:
            # 2. Find all paragraph tags (p) within the main element
            paragraph_elements = main_element.find_all(paragraph_tag)

            # 3. Extract, clean, and filter the text from each element
            paragraphs = [clean_text(p.get_text()) for p in paragraph_elements]

            # Filter out any resulting empty strings
            return [p for p in paragraphs if p]

    except Exception as e:
        print(f"❌ Error scraping {url}: {e}")
    return [] # Return an empty list on failure


def scrape_and_process_pair_strict(index, en_url, fa_url, selector, output_file, paragraph_tag='p'):
    """Fetches, aligns strictly by paragraph, and filters by sentence count."""

    # 1. Fetch Texts (now returns lists of paragraphs by HTML element)
    en_paragraphs = scrape_url_by_elements(en_url, selector, paragraph_tag)
    fa_paragraphs = scrape_url_by_elements(fa_url, selector, paragraph_tag)

    print(f"Talk #: {index}")
    print(f"Number of English Paragraphs: {len(en_paragraphs)}")
    print(f"Number of Persian Paragraphs: {len(fa_paragraphs)}")

    if not en_paragraphs or not fa_paragraphs:
        print(f"⚠️ Talk skipped: Could not retrieve or parse paragraphs for: {en_url}. Skipping talk.")
        return

    # 2. STRICT ALIGNMENT CHECK: Paragraph count
    if len(en_paragraphs) != len(fa_paragraphs):
        print(f"⚠️ Talk skipped: Paragraph count mismatch (EN {len(en_paragraphs)} vs FA {len(fa_paragraphs)}) for {en_url}.")
        return

    saved_sentences = 0
    discarded_paragraphs = 0

    with open(output_file, 'a', encoding='utf-8') as f_out:
        for en_para, fa_para in zip(en_paragraphs, fa_paragraphs):

            # 3. Segment Sentences within the aligned paragraphs
            en_sentences = split_text_into_sentences(en_para, lang='en')
            fa_sentences = split_text_into_sentences(fa_para, lang='fa')

            # 4. STRICT SENTENCE FILTER: Check for exact count match AND non-zero count
            if len(en_sentences) == len(fa_sentences) and len(en_sentences) > 0:

                # Save the perfectly aligned sentences
                for en, fa in zip(en_sentences, fa_sentences):
                    entry = {"english": en, "persian": fa}
                    f_out.write(json.dumps(entry, ensure_ascii=False) + '\n')
                    saved_sentences += 1
            else:
                # Discard the entire paragraph (due to mismatch or zero sentences)
                discarded_paragraphs += 1

    print(f"✅ Processed {en_url}. Saved {saved_sentences} pairs. Discarded {discarded_paragraphs} misaligned/empty paragraphs.")


### Run the scraper

Populate `talk_pairs` with your own list of `(english_url, persian_url)`
tuples. For a large list, consider moving this to a separate JSON/CSV file
and loading it here instead of hardcoding it in the notebook.


In [ ]:
# --- Example of Batch Usage ---
if __name__ == '__main__':
    # REMEMBER TO INSTALL: !pip install requests beautifulsoup4

    talk_pairs = [
        # Populate with your list of EN/FA URL pairs:
        # ("English_URL_1", "Persian_URL_1"),
        # ("English_URL_2", "Persian_URL_2"),
        # ...
    ]

    # Clear the file before starting a new run
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        pass

    for index, (en_url, fa_url) in enumerate(talk_pairs):
        scrape_and_process_pair_strict(index, en_url, fa_url, CONTENT_SELECTOR, OUTPUT_FILE, PARAGRAPH_TAG)

## 3. Build the Hugging Face Dataset

Load the scraped JSONL, reshape it into the `{"translation": {"en": ..., "fa": ...}}`
format expected by seq2seq tooling, and split into train/test.


In [44]:
from datasets import load_dataset, Dataset

raw_datasets = load_dataset('json', data_files='DATASET.jsonl', split='train')

def create_translation_dict(example):
    return {
        'translation': {
            'en': example['english'],
            'fa': example['persian']
        }
    }

transformed_datasets = raw_datasets.map(create_translation_dict)

final_datasets = transformed_datasets.remove_columns(['english', 'persian'])

train_test_split = final_datasets.train_test_split(test_size=0.1)

train = train_test_split['train']
test = train_test_split['test']

print(train_test_split)
# DatasetDict({
#     train: Dataset(...)
#     test: Dataset(...)
# })
print(train_test_split['train'][0])
# {'translation': {'en': '...', 'fa': '...'}}

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 43952
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 4884
    })
})
{'translation': {'en': 'This woodpile contains an enormous amount of fuel, capable of producing light and heat for days.', 'fa': 'این تودۀ عظیم چوبی دارای سوختی است که قادر به ایجاد حرارت و نور عظیمی برای چندین روز میباشد.'}}


## 4. Load Base Model & Tokenizer

In [45]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

model_name = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

### Sanity check: translate with the *un-fine-tuned* base model

In [46]:
text = "God is merciful and full of compassion."

# Prepare inputs
inputs = tokenizer(text, return_tensors="pt")

# Generate translation, forcing Persian as output language
generated_tokens = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.lang_code_to_id["fa_IR"]
)

persian = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
print(persian)

خداوند بخشنده و پر از شفقت است.


## 5. Fine-Tune

Tokenize the dataset, then fine-tune with `Seq2SeqTrainer`.

Key settings: effective batch size 8 (batch size 1 × 8 gradient accumulation
steps), learning rate 2e-5, 3 epochs, fp16, label smoothing 0.1 — conservative
choices suited to fine-tuning a large pretrained model on a single Colab GPU.


In [68]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

# Set source and target languages on the tokenizer
tokenizer.src_lang = "en_XX"
tokenizer.tgt_lang = "fa_IR"


def preprocess(batch):
    src = [x["en"] for x in batch["translation"]]
    tgt = [x["fa"] for x in batch["translation"]]

    model_inputs = tokenizer(
        src, text_target=tgt, max_length=256, truncation=True
    )

    return model_inputs


train_tokenized = train.map(
    preprocess, batched=True, remove_columns=train.column_names
)
test_tokenized = test.map(
    preprocess, batched=True, remove_columns=test.column_names
)

# Estimate total training steps
effective_batch_size = 4 * 8  # per_device_train_batch_size * gradient_accumulation_steps
steps_per_epoch = max(1, len(train_tokenized) // effective_batch_size)
total_steps = int(steps_per_epoch * 3)
print(
    f"Effective batch size: {effective_batch_size} | Steps/epoch:"
    f" {steps_per_epoch} | Estimated total steps: {total_steps}"
)

args = Seq2SeqTrainingArguments(
    output_dir="mbart-fa-religious-final",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    num_train_epochs=3,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    predict_with_generate=True,
    generation_num_beams=4,
    remove_unused_columns=True,
    label_smoothing_factor=0,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, label_pad_token_id=-100
)

fa_id = tokenizer.convert_tokens_to_ids("fa_IR")
model.generation_config.decoder_start_token_id = tokenizer.eos_token_id  # 2
model.generation_config.forced_bos_token_id = tokenizer.lang_code_to_id["fa_IR"]

model.config.forced_bos_token_id = None

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer,
)

Map:   0%|          | 0/43952 [00:00<?, ? examples/s]

Map:   0%|          | 0/4884 [00:00<?, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Effective batch size: 32 | Steps/epoch: 1373 | Estimated total steps: 4119


In [54]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,10.218759,1.240696
2,9.414924,1.177699
3,7.799063,1.171228


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=4122, training_loss=9.46598117088936, metrics={'train_runtime': 3637.1872, 'train_samples_per_second': 36.252, 'train_steps_per_second': 1.133, 'total_flos': 1.2127562933993472e+16, 'train_loss': 9.46598117088936, 'epoch': 3.0})

### Confirm the metrics of the saved (best) checkpoint

`load_best_model_at_end=True` means the trainer has already swapped in whichever evaluated checkpoint had the lowest `eval_loss`. This just prints its metrics clearly rather than digging them out of the logs.


In [56]:
final_metrics = trainer.evaluate()
print(final_metrics)


Training Loss,Validation Loss,Epoch
7.799063,1.171228,3


{'eval_loss': 1.171228051185608}


## 6. Save the Model

Save to Google Drive (persistent storage across Colab sessions) and push to
the Hugging Face Hub.


In [57]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [69]:
# Define your permanent save directory
save_directory = "/content/drive/MyDrive/Models/mbart-fa-religious-final"

# Save the model and tokenizer to Google Drive
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"Model successfully saved to: {save_directory}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model successfully saved to: /content/drive/MyDrive/Models/mbart-fa-religious-final


In [70]:
trainer.push_to_hub("mbart-fa-religious-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/tuckj90/mbart-fa-religious-final/commit/1ef3411caa01ef7b9763f293c03d0256f4c65fc9', commit_message='mbart-fa-religious-final', commit_description='', oid='1ef3411caa01ef7b9763f293c03d0256f4c65fc9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tuckj90/mbart-fa-religious-final', endpoint='https://huggingface.co', repo_type='model', repo_id='tuckj90/mbart-fa-religious-final'), pr_revision=None, pr_num=None)

## 7. Inference & Evaluation

Reload the fine-tuned model from the Hub and translate held-out test examples.


In [71]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

tokenizer = MBart50TokenizerFast.from_pretrained(
    "mbart-fa-religious-final",
    src_lang="en_XX"
)

model = MBartForConditionalGeneration.from_pretrained(
    "mbart-fa-religious-final"
).to("cuda")

tokenizer.tgt_lang = "fa_IR"


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

In [72]:
def translate(text, num_beams=4):
    encoded = tokenizer(text, return_tensors="pt").to("cuda")

    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.lang_code_to_id["fa_IR"],
        num_beams=num_beams,
    )

    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)


In [76]:
translate("God said, let there be light")


'خدا گفت، بگذار نور باشد'

### BLEU on the held-out test set

In [77]:
from evaluate import load

bleu = load("bleu")
results = bleu.compute(
    predictions=[translate(x["translation"]["en"]) for x in test],
    references=[[x["translation"]["fa"]] for x in test]
)
print(results)

{'bleu': 0.26273285003844005, 'precisions': [0.5806799057556379, 0.3266162803604404, 0.19947358925103212, 0.12594979912937412], 'brevity_penalty': 1.0, 'length_ratio': 1.0464636502697044, 'translation_length': 103985, 'reference_length': 99368}
